# FlyTrack 離線 ML：Colab 操作範本
選 GPU runtime 後依序執行。Colab 只做離線訓練；Gazebo／ROS 展示留在 Linux 主機。子圖與 flyvis 前端已在 repo 中，不必重新下載原始資料。
每個階段結束後會把 `runs/` 同步到 Google Drive 的 `MyDrive/flytrack/`。

In [ ]:
REPO_URL = 'https://github.com/<owner>/<repo>.git'  # 改成組內 repo
BRANCH = 'main'

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys
drive_root = Path('/content/drive/MyDrive/flytrack')
drive_root.mkdir(parents=True, exist_ok=True)
project = Path('/content/flytrack')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'rev-parse', 'HEAD'], check=True)

## 隔離 Python 3.12 與鎖定依賴
Notebook kernel 不必等於訓練 Python；後續命令都透過獨立的 Python 3.12 venv 執行。

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.13.1'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.12', str(project / '.venv')], check=True)
python = str(project / '.venv/bin/python')
def run(*args):
    subprocess.run(list(args), cwd=project, check=True)
run('uv', 'pip', 'install', '--python', python, '--require-hashes', '-r', 'requirements/ml-cu126-py312.lock')
run('uv', 'pip', 'install', '--python', python, '--no-deps', '-e', '.')
run('nvidia-smi')
run(python, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())')

## 確認環境
應為 66 passed；接著跑約 10 秒的 PPO 小流程。

In [ ]:
run(python, '-m', 'pytest', '-q', 'python/tests')
run(python, '-m', 'flytrack_train', 'ppo', '--config', 'configs/train/smoke.yaml', '--device', 'cuda')

## 產生實驗設定並跑一個變體／種子
`SEED` 設成要跑的訓練 seed。先跑單一變體確認速度與顯存，再跑完整的 `run_all.sh`。

In [ ]:
SEED = 0
VARIANT = 'VO'
run(python, 'scripts/prepare_experiments.py', '--algorithm', 'ppo', '--device', 'cuda', '--seeds', str(SEED))
commands = [line.split() for line in (project / 'runs/experiment_configs/run_all.sh').read_text().splitlines()
            if line.startswith('python ') and f'/{VARIANT}/seed_{SEED}/' in line]
for command in commands:
    try:
        run(python, *command[1:])
    finally:
        shutil.copytree(project / 'runs', drive_root / 'runs', dirs_exist_ok=True)

中斷後重新執行前兩格與上一格即可；已完成的階段可在 `commands` 中略過，PPO 中途中斷可在命令後加 `--resume`。跑完把 Drive 上的 `runs/<variant>/seed_<n>/` 交回整合。